In [20]:
import os

In [21]:
%pwd

'C:\\Users\\Amanda\\Desktop\\Text-Summarizer\\Text-Summarizer-Project'

In [22]:
os.chdir("C:/Users/Amanda/Desktop/Text-Summarizer/Text-Summarizer-Project")

In [23]:
%pwd

'C:\\Users\\Amanda\\Desktop\\Text-Summarizer\\Text-Summarizer-Project'

In [24]:
from  dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file_name: Path
    

In [25]:
%pip install python-box ensure pyYAML joblib
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "src")))
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

Note: you may need to restart the kernel to use updated packages.


In [26]:
from pathlib import Path

class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        
        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_path=config.model_path,
            tokenizer_path=config.tokenizer_path,
            metric_file_name=config.metric_file_name
        )

        return model_evaluation_config
    

In [27]:
! pip install transformers datasets sentencepiece accelerate
! pip install evaluate
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM
from datasets import load_dataset,load_from_disk

import evaluate
import torch
import pandas as pd
from tqdm import tqdm


In [31]:
! pip install rouge_score nltk


  Using cached rouge_score-0.1.2-py3-none-any.whl
  Using cached nltk-3.9.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
Using cached nltk-3.9.2-py3-none-any.whl (1.5 MB)
Using cached click-8.3.1-py3-none-any.whl (108 kB)

   ---------------------------------------- 0/3 [click]
   ---------------------------------------- 0/3 [click]
   ------------- -------------------------- 1/3 [nltk]
   ------------- -------------------------- 1/3 [nltk]
   ------------- -------------------------- 1/3 [nltk]
   ------------- -------------------------- 1/3 [nltk]
   ------------- -------------------------- 1/3 [nltk]
   ------------- -------------------------- 1/3 [nltk]
   ------------- -------------------------- 1/3 [nltk]
   ------------- -------------------------- 1/3 [nltk]
   ------------- -------------------------- 1/3 [nltk]
   ------------- -------------------------- 1/3 [nltk]
   ------------- -------------------------- 1/3 [nltk]
   -----

In [32]:
import nltk
nltk.download("punkt")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Amanda\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [33]:
from pathlib import Path
import torch
import pandas as pd
from tqdm import tqdm
from datasets import load_from_disk
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


class ModelEvaluation:
    def __init__(self, config):
        self.config = config

    def generate_batch_sized_chunks(self, list_of_elements, batch_size):
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i:i + batch_size]

    def calculate_metric_on_test_ds(
        self,
        dataset,
        metric,
        model,
        tokenizer,
        batch_size=16,
        device=None,
        column_text="dialogue",
        column_summary="summary"
    ):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"

        article_batches = list(
            self.generate_batch_sized_chunks(dataset[column_text], batch_size)
        )
        target_batches = list(
            self.generate_batch_sized_chunks(dataset[column_summary], batch_size)
        )

        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches),
            total=len(target_batches)
        ):
            inputs = tokenizer(
                article_batch,
                max_length=1024,
                truncation=True,
                padding="max_length",
                return_tensors="pt"
            )

            summaries = model.generate(
                input_ids=inputs["input_ids"].to(device),
                attention_mask=inputs["attention_mask"].to(device),
                max_length=128,
                num_beams=8,
                length_penalty=0.8
            )

            decoded_summaries = [
                tokenizer.decode(s, skip_special_tokens=True)
                for s in summaries
            ]

            metric.add_batch(
                predictions=decoded_summaries,
                references=target_batch
            )

        return metric.compute()

    def evaluate(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"

        tokenizer = AutoTokenizer.from_pretrained(
            Path(self.config.tokenizer_path),
            local_files_only=True
        )

        model = AutoModelForSeq2SeqLM.from_pretrained(
            Path(self.config.model_path),
            local_files_only=True
        ).to(device)

        dataset = load_from_disk(Path(self.config.data_path))

        rouge_metric = evaluate.load("rouge")

        score = self.calculate_metric_on_test_ds(
            dataset["test"][:10],
            rouge_metric,
            model,
            tokenizer,
            batch_size=2
        )

        rouge_dict = {
            k: score[k] for k in ["rouge1", "rouge2", "rougeL", "rougeLsum"]
        }

        pd.DataFrame(rouge_dict, index=["pegasus"]).to_csv(
            self.config.metric_file_name,
            index=False
        )


In [34]:
try:
    config_manager = ConfigurationManager()
    model_evaluation_config = config_manager.get_model_evaluation_config()

    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.evaluate()


except Exception as e:
    raise e

[2026-01-04 08:26:48,060]: INFO:common: yaml file: C:\Users\Amanda\Desktop\Text-Summarizer\Text-Summarizer-Project\config\config.yaml loaded successfully
[2026-01-04 08:26:48,062]: INFO:common: yaml file: C:\Users\Amanda\Desktop\Text-Summarizer\Text-Summarizer-Project\params.yaml loaded successfully
[2026-01-04 08:26:48,063]: INFO:common: created directory at: artifacts
[2026-01-04 08:26:48,065]: INFO:common: created directory at: artifacts/model_evaluation


100%|██████████| 5/5 [07:03<00:00, 84.68s/it] 

[2026-01-04 08:33:54,832]: INFO:rouge_scorer: Using default tokenizer.
